# Praktikum Sub-CPMK-6 &mdash; Morfologi Citra Biner untuk Perbaikan Hasil Pemrosesan

**Pengolahan Citra Digital (IF2514505)** &nbsp;|&nbsp; Semester 5 &nbsp;|&nbsp; 1 SKS Praktikum &nbsp;|&nbsp; Durasi sasaran 135 menit (rentang 120&ndash;150)

**Program Studi Informatika &mdash; Institut Teknologi Kalimantan**

---

| | |
|---|---|
| **Nama** | ................................................ |
| **NIM** | ................................................ |
| **Kelas / Kelompok** | ................................................ |
| **Tanggal praktikum** | ................................................ |

---

### Capaian yang diukur
**Sub-CPMK-6**: Mahasiswa mampu menerapkan operasi morfologi pada citra biner untuk perbaikan hasil pemrosesan.

### Rujukan
Gonzalez, R.C. dan Woods, R.E. (2018) *Digital Image Processing*, edisi ke-4, Bab 9 (§9.1&ndash;9.7). Nomor persamaan mengikuti buku tersebut.

### Petunjuk pengerjaan
1. Jalankan sel berurutan. Sel bertanda `# TODO` wajib Anda lengkapi sendiri. Fungsi yang Anda tulis dipakai lagi di bagian berikutnya.
2. Setiap blok **Analisis** diisi kalimat Anda sendiri dan **wajib memuat angka** dari keluaran Anda.
3. Citra uji dibangkitkan di notebook (benih tetap). **Tidak perlu mengunggah berkas.**
4. Citra biner berupa larik `bool`: **True = latar depan (objek, putih)**. Di luar citra dianggap latar belakang, sesuai Gonzalez §9.1.
5. SE ditulis sebagai larik bernilai 1 (latar depan), 0 (latar belakang), &minus;1 (tak peduli). Asal SE di tengah larik.
6. Bagian **[Pengayaan]** boleh dilewati; jalur inti selesai dalam &plusmn;120 menit.
7. Kumpulkan dalam keadaan **sudah dijalankan penuh** dengan nama `PraktikumPCD_SubCPMK6_NIM_Nama.ipynb`.

| Bagian | Isi | Menit |
|---|---|---|
| 0 | Persiapan dan citra uji | 10 |
| 1 | Erosi dan dilasi dari definisi; jebakan pustaka (§9.1&ndash;9.2) | 20 |
| 2 | Opening, closing, dan memilih elemen penstruktur (§9.3) | 25 |
| 3 | Hit-or-miss, batas, isi lubang, komponen terhubung (§9.4&ndash;9.5) | 30 |
| 4 | Kerangka dan rekonstruksi morfologi (§9.5&ndash;9.6) | 25 |
| 5 | Studi kasus: membersihkan masker deteksi kapal dari citra SAR | 20 |
| 6 | Pemeriksaan akhir dan pengumpulan | 5 |


## Bagian 0 &mdash; Persiapan (10 menit)
Jalankan sel berikut **tanpa diubah**.

In [ ]:
# Sel persiapan — jalankan tanpa diubah
import numpy as np
import cv2
import matplotlib.pyplot as plt
from scipy import ndimage
from skimage import morphology as skm

plt.rcParams['figure.dpi'] = 110
print('NumPy', np.__version__, '| OpenCV', cv2.__version__)

"""Pembangkit citra uji sintetis Sub-CPMK-6 (benih acak tetap, hanya NumPy).

Semua citra biner dikembalikan sebagai larik bool (True = latar depan/objek).
Huruf memakai fon bitmap 5x7 yang didefinisikan di sini, sehingga citra identik di semua versi pustaka.
"""
import numpy as np

BENIH = 2027

_FON = {
    "A": ".###.|#...#|#...#|#####|#...#|#...#|#...#", "B": "####.|#...#|#...#|####.|#...#|#...#|####.",
    "C": ".###.|#...#|#....|#....|#....|#...#|.###.", "D": "####.|#...#|#...#|#...#|#...#|#...#|####.",
    "E": "#####|#....|#....|####.|#....|#....|#####", "F": "#####|#....|#....|####.|#....|#....|#....",
    "G": ".###.|#...#|#....|#.###|#...#|#...#|.###.", "H": "#...#|#...#|#...#|#####|#...#|#...#|#...#",
    "I": ".###.|..#..|..#..|..#..|..#..|..#..|.###.", "K": "#...#|#..#.|#.#..|##...|#.#..|#..#.|#...#",
    "L": "#....|#....|#....|#....|#....|#....|#####", "M": "#...#|##.##|#.#.#|#.#.#|#...#|#...#|#...#",
    "N": "#...#|#...#|##..#|#.#.#|#..##|#...#|#...#", "O": ".###.|#...#|#...#|#...#|#...#|#...#|.###.",
    "P": "####.|#...#|#...#|####.|#....|#....|#....", "R": "####.|#...#|#...#|####.|#.#..|#..#.|#...#",
    "S": ".####|#....|#....|.###.|....#|....#|####.", "T": "#####|..#..|..#..|..#..|..#..|..#..|..#..",
    "U": "#...#|#...#|#...#|#...#|#...#|#...#|.###.", "Y": "#...#|#...#|.#.#.|..#..|..#..|..#..|..#..",
    "0": ".###.|#...#|#..##|#.#.#|##..#|#...#|.###.", "1": "..#..|.##..|..#..|..#..|..#..|..#..|.###.",
    "2": ".###.|#...#|....#|...#.|..#..|.#...|#####", "3": "#####|...#.|..#..|...#.|....#|#...#|.###.",
    "4": "...#.|..##.|.#.#.|#..#.|#####|...#.|...#.", "5": "#####|#....|####.|....#|....#|#...#|.###.",
    "6": "..##.|.#...|#....|####.|#...#|#...#|.###.", "7": "#####|....#|...#.|..#..|.#...|.#...|.#...",
    "8": ".###.|#...#|#...#|.###.|#...#|#...#|.###.", "9": ".###.|#...#|#...#|.####|....#|...#.|.##..",
    " ": ".....|.....|.....|.....|.....|.....|.....",
}
GLIF = {k: np.array([[c == "#" for c in r] for r in v.split("|")], bool) for k, v in _FON.items()}


def _grid(h, w):
    return np.mgrid[0:h, 0:w].astype(np.float64)


def tulis(img, teks, baris, kolom, skala, jarak=1):
    """Menulis teks (latar depan True) mulai (baris, kolom); jarak antarhuruf = jarak titik fon."""
    langkah = (5 + jarak) * skala
    for i, ch in enumerate(teks):
        g = np.kron(GLIF[ch], np.ones((skala, skala), bool))
        b0, k0 = baris, kolom + i * langkah
        img[b0:b0 + g.shape[0], k0:k0 + g.shape[1]] |= g
    return img


# --------------------------------------------------------------- contoh hitung tangan
def contoh_kecil():
    """Persegi panjang 3x5 (15 piksel) di dalam citra 7x9 — contoh hitung tangan erosi dan dilasi."""
    A = np.zeros((7, 9), bool)
    A[2:5, 2:7] = True
    return A


def contoh_L():
    """Persegi 5x5 dan huruf 'L' berlengan selebar 1 piksel (citra 9x16) — contoh opening."""
    A = np.zeros((9, 16), bool)
    A[2:7, 2:7] = True                 # persegi 5x5
    A[1:8, 10] = True                  # lengan tegak L (lebar 1)
    A[7, 10:15] = True                 # lengan datar L (lebar 1)
    return A


# --------------------------------------------------------------- objek biner berderau
def objek_biner(h=256, w=256):
    """Tujuh objek: persegi, cakram, segitiga, cincin (berlubang), kapal bertiang tipis, elips, persegi kecil."""
    yy, xx = _grid(h, w)
    A = np.zeros((h, w), bool)
    A[30:80, 25:95] = True                                               # persegi panjang
    A[(yy - 60) ** 2 + (xx - 175) ** 2 <= 32 ** 2] = True                # cakram
    A[(yy >= 115) & (yy <= 175) & (xx >= 20) & (xx <= 20 + (yy - 115) * 1.2)] = True   # segitiga
    cin = (yy - 145) ** 2 + (xx - 140) ** 2
    A[(cin <= 30 ** 2) & (cin >= 16 ** 2)] = True                        # cincin, lubang berjari-jari 16
    A[200:226, 160:236] = True                                           # lambung kapal
    A[170:200, 196:199] = True                                           # tiang selebar 3 piksel
    A[((yy - 215) / 14.0) ** 2 + ((xx - 70) / 34.0) ** 2 <= 1] = True    # elips
    A[118:124, 212:218] = True                                           # persegi kecil 6x6
    return A


def beri_derau_biner(A, rng, p=0.03):
    """Membalik proporsi p piksel secara acak (derau garam-merica biner)."""
    return A ^ (rng.random(A.shape) < p)


# --------------------------------------------------------------- teks biner
JUDUL = "MANIFES"
BADAN = ["KM NUSA BAHARI 07", "01 ARIF  02 SINTA", "03 BUDI  04 RINA"]


def teks_biner(h=200, w=360):
    """Judul besar (skala 6, goresan 6 px) dan badan teks kecil (skala 2, goresan 2 px)."""
    A = np.zeros((h, w), bool)
    tulis(A, JUDUL, 12, 20, 6)
    for i, t in enumerate(BADAN):
        tulis(A, t, 92 + i * 30, 20, 2, jarak=2)   # celah antarhuruf 4 piksel
    J = np.zeros((h, w), bool)
    tulis(J, JUDUL, 12, 20, 6)                 # masker kebenaran judul saja
    return A, J


def teks_putus(A, rng, n=40, baris_mulai=88):
    """Memutus goresan badan teks: n potongan tegak/datar selebar 1 piksel dan sepanjang 5 piksel."""
    B = A.copy()
    ys, xs = np.nonzero(A[baris_mulai:])
    ys = ys + baris_mulai
    pilih = rng.choice(len(ys), n, replace=False)
    for k, i in enumerate(pilih):
        y, x = ys[i], xs[i]
        if k % 2 == 0:
            B[y - 2:y + 3, x] = False
        else:
            B[y, x - 2:x + 3] = False
    return B


# --------------------------------------------------------------- cincin tipis (jebakan konektivitas)
def cincin_tipis(h=64, w=160):
    """Tiga kontur setebal 1 piksel: persegi (terhubung-4), belah ketupat dan lingkaran (hanya terhubung-8)."""
    A = np.zeros((h, w), bool)
    A[12, 10:41] = A[52, 10:41] = True; A[12:53, 10] = A[12:53, 40] = True     # persegi
    for t in range(21):                                                     # belah ketupat
        A[32 - 20 + t, 80 + t] = A[32 + t, 100 - t] = True
        A[32 + 20 - t, 80 - t] = A[32 - t, 60 + t] = True
    yy, xx = _grid(h, w)
    r = np.hypot(yy - 32, xx - 132)
    lingk = np.abs(r - 18) < 0.5
    A |= lingk
    return A


# --------------------------------------------------------------- kerangka, cembung
def siluet_kapal(h=120, w=220):
    """Siluet kapal (lambung trapesium, anjungan, cerobong) untuk kerangka dan pemangkasan."""
    yy, xx = _grid(h, w)
    A = (yy >= 70) & (yy <= 100) & (xx >= 15 + (yy - 70) * 0.8) & (xx <= 205 - (yy - 70) * 1.2)
    A |= (yy >= 40) & (yy < 70) & (xx >= 120) & (xx <= 170)
    A |= (yy >= 18) & (yy < 40) & (xx >= 140) & (xx <= 152)
    return A


def bentuk_cekung(h=80, w=80):
    """Bentuk cekung seperti bulan sabit untuk selubung cembung."""
    yy, xx = _grid(h, w)
    A = ((yy - 40) ** 2 + (xx - 38) ** 2 <= 30 ** 2) & ~((yy - 34) ** 2 + (xx - 54) ** 2 <= 24 ** 2)
    return A


# --------------------------------------------------------------- adegan SAR (studi kasus)
KAPAL = [  # (pusat baris, pusat kolom, setengah panjang, setengah lebar, sudut derajat)
    (60, 90, 20, 5, 15), (150, 70, 14, 4, -30), (110, 230, 24, 6, 80), (200, 180, 10, 3, 0),
    (190, 300, 16, 4, 45), (45, 250, 4, 2, 0), (226, 92, 12, 4, 0), (235, 92, 12, 4, 0),   # dua kapal bersandingan (bersentuhan)
    (128, 377, 14, 5, 0),                                                                   # kapal terpotong tepi kanan
]


def sar_kapal(rng, h=256, w=384, looks=3, ambang=2.6):
    """Citra intensitas SAR sintetis berspekel (multiplikatif, Gamma), kapal terang, daratan di pojok kiri atas.

    Mengembalikan (intensitas float, biner hasil ambang, masker kapal, daftar masker per kapal, masker darat).
    Ambang = ambang * rerata laut (gaya CFAR global sederhana).
    """
    yy, xx = _grid(h, w)
    R = np.ones((h, w))
    darat = ((yy / 70.0) ** 2 + (xx / 110.0) ** 2 <= 1) | ((yy < 12) & (xx < 160))
    R[darat] = (5.0 + 1.5 * np.sin(xx / 7.0) * np.cos(yy / 9.0))[darat]
    per_kapal = []
    for cy, cx, a, b, t in KAPAL:
        tr = np.deg2rad(t)
        u = (xx - cx) * np.cos(tr) + (yy - cy) * np.sin(tr)
        v = -(xx - cx) * np.sin(tr) + (yy - cy) * np.cos(tr)
        m = (np.abs(u) <= a) & (np.abs(v) <= b) & ((np.abs(u) <= a - b) | ((np.abs(u) - (a - b)) ** 2 + v ** 2 <= b ** 2))
        R[m] = 9.0
        per_kapal.append(m)
    spekel = rng.gamma(looks, 1.0 / looks, (h, w))
    I = R * spekel
    biner = I > ambang * np.mean(I[~darat & ~np.any(per_kapal, axis=0)])
    kapal = np.any(per_kapal, axis=0)
    return I, biner, kapal, per_kapal, darat


def bangkitkan_semua():
    rng = np.random.default_rng(BENIH)
    obj = objek_biner()
    obj_d = beri_derau_biner(obj, rng)
    teks, judul = teks_biner()
    putus = teks_putus(teks, rng)
    I, sar, kapal, per_kapal, darat = sar_kapal(rng)
    return {"kecil": contoh_kecil(), "L": contoh_L(), "objek": obj, "objek_derau": obj_d,
            "teks": teks, "judul": judul, "teks_putus": putus, "cincin": cincin_tipis(),
            "kapal_siluet": siluet_kapal(), "cekung": bentuk_cekung(),
            "sar_I": I, "sar": sar, "sar_kapal": kapal, "sar_per_kapal": per_kapal, "sar_darat": darat}

KOTAK3 = np.ones((3, 3), int)
SILANG = np.array([[0, 1, 0], [1, 1, 1], [0, 1, 0]])

def elips(n):
    return cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (n, n)).astype(int)

def geser(A, dy, dx, isi=False):
    """B(y, x) = A(y + dy, x + dx); piksel dari luar citra diisi `isi`."""
    h, w = A.shape
    B = np.full((h, w), isi, bool)
    ys, ye = max(0, -dy), min(h, h - dy)
    xs, xe = max(0, -dx), min(w, w - dx)
    if ys < ye and xs < xe:
        B[ys:ye, xs:xe] = A[ys + dy:ye + dy, xs + dx:xe + dx]
    return B

def titik_se(B, nilai=1):
    """Daftar pergeseran (dy, dx) relatif terhadap asal di tengah, untuk elemen SE yang bernilai `nilai`."""
    B = np.asarray(B); oy, ox = B.shape[0] // 2, B.shape[1] // 2
    return [(i - oy, j - ox) for i, j in zip(*np.nonzero(B == nilai))]

def pantul(B):
    """Eq. 9-1: rotasi 180° terhadap asal."""
    return np.asarray(B)[::-1, ::-1]

def nkomp(X, konek=8):
    return int(ndimage.label(X, np.ones((3, 3)) if konek == 8 else SILANG)[1])

def ujung(X):
    """Banyaknya titik ujung (piksel latar depan dengan tepat satu tetangga-8)."""
    n = ndimage.convolve(X.astype(int), np.ones((3, 3), int), mode='constant') - X
    return int((X & (n == 1)).sum())

def cetak(A):
    for r in np.asarray(A): print(''.join('#' if v else '.' for v in r))

def tampil(daftar, judul=None, kolom=None, tinggi=3.0):
    n = len(daftar); kolom = kolom or n; baris = int(np.ceil(n / kolom))
    fig, ax = plt.subplots(baris, kolom, figsize=(3.8 * kolom, tinggi * baris)); ax = np.atleast_1d(ax).ravel()
    for a in ax: a.axis('off')
    for a, im, t in zip(ax, daftar, judul or [''] * n):
        a.imshow(im, cmap='gray', interpolation='nearest'); a.set_title(t, fontsize=9)
    plt.tight_layout(); plt.show()

C = bangkitkan_semua()
kecil, L_, objek, objek_d = C['kecil'], C['L'], C['objek'], C['objek_derau']
teks, judul, teks_p, cincin = C['teks'], C['judul'], C['teks_putus'], C['cincin']
siluet, cekung = C['kapal_siluet'], C['cekung']
sar_I, sar, per_kapal, darat = C['sar_I'], C['sar'], C['sar_per_kapal'], C['sar_darat']
tampil([objek, objek_d, teks, teks_p, cincin, siluet, np.clip(sar_I / 12, 0, 1), sar],
       ['objek', 'objek berderau', 'teks', 'teks terputus', 'cincin tipis', 'siluet kapal', 'SAR (intensitas)', 'SAR (ambang)'], kolom=4)

## Bagian 1 &mdash; Erosi dan dilasi dari definisi (20 menit)

Erosi (Gonzalez Eq. 9-3): $A \ominus B = \{z \mid (B)_z \subseteq A\}$, artinya hasil di $z$ bernilai 1 bila **semua** $A(z + b)$ bernilai 1 untuk setiap $b \in B$.

Dilasi (Eq. 9-6): $A \oplus B = \{z \mid (\hat B)_z \cap A \neq \varnothing\}$, artinya hasil di $z$ bernilai 1 bila **ada** $b \in B$ dengan $A(z - b)$ bernilai 1.

Gunakan `titik_se(B)` untuk daftar $b$ dan `geser(A, dy, dx)` untuk $A(z + b)$. Jangan memakai fungsi morfologi pustaka di dalam fungsi Anda.

In [ ]:
# TODO 1.1 — Lengkapi erosi dan dilasi. Parameter `isi` = nilai piksel di luar citra (Gonzalez: latar belakang = False).
def erosi(A, B, isi=False):
    ...

def dilasi(A, B):
    ...

# Uji 1: hitung tangan persegi panjang 3x5 (15 piksel) dengan kotak 3x3 -> erosi 3 piksel, dilasi 35 piksel
# print(kecil.sum(), erosi(kecil, KOTAK3).sum(), dilasi(kecil, KOTAK3).sum())
# Uji 2: bandingkan dengan SciPy pada citra acak dan SE tak simetris (harus True, True)
rng = np.random.default_rng(0)
X = rng.random((60, 80)) < 0.4
Bq = np.array([[0, 1, 1], [0, 1, 0], [1, 0, 0]])


In [ ]:
# TODO 1.2 — Tiga jebakan pustaka.
# (a) SE tak simetris B_as = {asal, tetangga kanan}. Dilasikan impuls di (3, 4) dengan dilasi Anda, cv2.dilate, dan
#     ndimage.binary_dilation; cetak koordinat hasilnya. Lalu hitung banyak piksel berbeda antara cv2.dilate(X, Bq) dan dilasi(X, Bq),
#     dan periksa apakah cv2.dilate(X, Bq) sama dengan dilasi(X, pantul(Bq)).
# (b) Kernel genap: opening persegi 8x8 dengan cv2.getStructuringElement(cv2.MORPH_RECT, (4, 4)) lewat cv2.morphologyEx.
#     Hitung piksel hasil yang berada DI LUAR persegi dan piksel persegi yang HILANG.
# (c) Tepi citra: erosi kotak 3x3 pada `pita` (menempel di tepi kiri, atas, bawah) dengan erosi Anda, ndimage.binary_erosion,
#     cv2.erode, dan skm.erosion. Cetak banyak piksel hasil masing-masing.
B_as = np.zeros((3, 3), int); B_as[1, 1] = B_as[1, 2] = 1
imp = np.zeros((7, 9), bool); imp[3, 4] = True
S8 = np.zeros((20, 20), np.uint8); S8[5:13, 6:14] = 1
pita = np.zeros((10, 10), bool); pita[:, :4] = True


**Analisis 1.** (a) Jelaskan hasil hitung tangan 3 dan 35 piksel dari ukuran persegi panjang dan SE. (b) Apa yang sebenarnya dihitung `cv2.dilate` bila SE tidak simetris, dan mengapa perbedaannya lenyap untuk kotak, silang, dan elips berukuran ganjil? Apa akibatnya pada opening dengan kernel 4&times;4? (c) Mengapa keempat implementasi erosi memberi dua jawaban berbeda pada pita yang menempel di tepi? Apakah salah satunya keliru?

*Jawaban:* ...

## Bagian 2 &mdash; Opening, closing, dan memilih elemen penstruktur (25 menit)

Opening (Eq. 9-10): $A \circ B = (A \ominus B) \oplus B$. Closing (Eq. 9-11): $A \bullet B = (A \oplus B) \ominus B$.

Sifat (Gonzalez §9.3): opening adalah himpunan bagian A; closing memuat A; keduanya idempoten. Agar closing tidak mengikis objek yang menempel di tepi, **lapisi citra dengan latar belakang** selebar ukuran SE, kerjakan, lalu potong kembali (Gonzalez §9.1).

In [ ]:
# TODO 2.1 — Lengkapi opening dan closing (dengan dan tanpa pelapis). Lalu periksa pada objek_d dengan KOTAK3:
#            idempoten (keduanya), opening ⊆ A, A ⊆ closing. Terakhir, hitung piksel `cl` yang HILANG oleh
#            closing Anda tanpa lapis, closing Anda dengan lapis, ndimage.binary_closing, dan cv2.morphologyEx(MORPH_CLOSE).
def opening(A, B):
    ...

def closing(A, B, lapis=True):
    ...

cl = np.zeros((12, 12), bool); cl[:, 0:3] = True; cl[5, 3:6] = True


In [ ]:
# TODO 2.2 — Pembersihan derau biner. objek_d = objek dengan 3 % piksel dibalik.
#   Untuk SE silang 3x3, kotak 3x3, elips 5x5, kotak 5x5, elips 7x7 dan dua urutan (opening→closing dan closing→opening), hitung:
#   jumlah komponen-8, % piksel salah terhadap `objek`, fraksi tiang kapal yang tersisa (masker `tiang`),
#   fraksi persegi kecil 6x6 yang tersisa (masker `kotak_kecil`), dan fraksi lubang cincin yang tetap terbuka (masker `lubang`).
print('acuan', nkomp(objek), 'komponen; berderau', nkomp(objek_d), 'komponen,', round(100 * (objek ^ objek_d).mean(), 3), '% salah')
tiang = np.zeros_like(objek); tiang[170:200, 196:199] = True
kotak_kecil = np.zeros_like(objek); kotak_kecil[118:124, 212:218] = True
yy, xx = np.mgrid[0:256, 0:256]; lubang = (yy - 145) ** 2 + (xx - 140) ** 2 < 16 ** 2
SE = {'silang3': SILANG, 'kotak3': KOTAK3, 'elips5': elips(5), 'kotak5': np.ones((5, 5), int), 'elips7': elips(7)}


In [ ]:
# TODO 2.3 — Memperbaiki teks terputus (bandingkan Gonzalez Contoh 9.2). teks_p = teks dengan 40 potongan selebar 1 piksel.
#   Untuk SE silang dan kotak 3x3: dilasi saja vs closing. Laporkan jumlah komponen (acuan: 46), piksel yang DITAMBAHKAN di luar `teks`,
#   dan piksel hilang yang KEMBALI. Tampilkan potongan baris badan teks.
print('teks', nkomp(teks), 'komponen; terputus', nkomp(teks_p), 'komponen;', int((teks & ~teks_p).sum()), 'piksel hilang dari', int(teks.sum()))


**Analisis 2.** (a) SE dan urutan mana yang Anda pilih untuk `objek_d`? Justifikasi dengan **dua** ukuran sekaligus. (b) Elips 7&times;7 opening&rarr;closing menghasilkan tepat 7 komponen. Apakah itu berarti pembersihan berhasil? Buktikan dengan daftar luas komponen. (c) Mengapa closing&rarr;opening dengan kotak 3&times;3 menyisakan komponen lebih banyak daripada opening&rarr;closing? (d) Untuk teks terputus, dilasi memulihkan jumlah komponen dengan sempurna. Mengapa closing tetap lebih tepat untuk tahap sebelum pembaca huruf? (e) Apa akibatnya bila Anda memakai `ndimage.binary_closing` begitu saja pada masker yang objeknya menempel di tepi citra?

*Jawaban:* ...

## Bagian 3 &mdash; Hit-or-miss, batas, isi lubang, komponen terhubung (30 menit)

Hit-or-miss dengan SE campuran (Eq. 9-17): $I \circledast B = \{z \mid (B)_z \subseteq I\}$, dengan elemen 1 harus cocok dengan latar depan, elemen 0 dengan latar belakang, dan &minus;1 bebas. Setara dengan $(A \ominus B_1) \cap (A^c \ominus B_2)$ (Eq. 9-16).

Batas (Eq. 9-18): $\beta(A) = A - (A \ominus B)$. Isi lubang dengan benih (Eq. 9-19): $X_k = (X_{k-1} \oplus B) \cap I^c$. Komponen terhubung (Eq. 9-20): $X_k = (X_{k-1} \oplus B) \cap I$.

In [ ]:
# TODO 3.1 — Lengkapi hmt(I, B). Petunjuk: erosi I dengan elemen bernilai 1 DAN erosi ~I dengan elemen bernilai 0.
#            Di luar citra = latar belakang, jadi bagi ~I nilai luarnya True (pakai parameter isi).
#            Uji: lubang satu piksel harus di (4, 4) dan (6, 7); sudut kanan atas di (2, 9).
def hmt(I, B):
    ...

Iu = np.zeros((12, 14), bool); Iu[2:9, 2:10] = True; Iu[4, 4] = False; Iu[6, 7] = False
SE_lubang = np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]])
SE_sudut = np.array([[0, 0, 0], [1, 1, 0], [1, 1, 0]])


In [ ]:
# TODO 3.2 — Batas (Eq. 9-18) pada cakram berjari-jari 40 dengan SE kotak 3x3 dan silang. Bandingkan banyak piksel batas
#            dengan keliling teoretis 2πr. Bandingkan juga dengan gradien morfologi cv2.MORPH_GRADIENT (kotak 3x3).
def batas(A, B=KOTAK3):
    ...

yy, xx = np.mgrid[0:101, 0:101]; cakram = (yy - 50) ** 2 + (xx - 50) ** 2 <= 40 ** 2


In [ ]:
# TODO 3.3 — Isi lubang dengan benih (Eq. 9-19) pada `cincin` (kontur setebal 1 piksel: persegi, belah ketupat, lingkaran).
#            Benih di (32, 80), di dalam belah ketupat. Jalankan dengan B = SILANG (seperti Gonzalez Gbr. 9.17c) dan B = KOTAK3.
#            Laporkan piksel yang terisi dan banyak iterasi. Bandingkan dengan banyak piksel latar belakang seluruh citra.
def isi_lubang_benih(I, benih, B=SILANG):
    """Mengembalikan (I ∪ X_k, banyak iterasi)."""
    ...


In [ ]:
# TODO 3.4 — Komponen terhubung dari satu titik (Eq. 9-20) dengan B = KOTAK3, mulai dari titik (50, 50) pada `objek`.
#            Laporkan luas dan banyak iterasi. Lalu hitung jumlah komponen `objek` dan `objek_d` dengan cv2.connectedComponents
#            untuk konektivitas 8 dan 4.
def komponen_dari_titik(I, titik, B=KOTAK3):
    """Mengembalikan (komponen, banyak iterasi)."""
    ...


In [ ]:
# [Pengayaan] TODO 3.5 — Selubung cembung (Eq. 9-21, 9-22) dengan empat SE Gbr. 9.21(a) (diberikan). Bandingkan luasnya dengan
#            skm.convex_hull_image(cekung), tanpa dan dengan pembatasan pada kotak pembatas objek.
def _putar_cincin(B, langkah):
    pos = [(0, 0), (0, 1), (0, 2), (1, 2), (2, 2), (2, 1), (2, 0), (1, 0)]
    B = np.asarray(B); Cc = B.copy()
    for i, p in enumerate(pos):
        Cc[pos[(i + langkah) % 8]] = B[p]
    return Cc
SE_CEMBUNG = [_putar_cincin([[1, -1, -1], [1, 0, -1], [1, -1, -1]], 2 * k) for k in range(4)]


**Analisis 3.** (a) Mengapa hit-or-miss memerlukan elemen latar belakang? Apa yang terdeteksi bila SE lubang hanya berisi elemen 1? (b) Bandingkan banyak piksel batas dari kotak 3&times;3 dan silang dengan keliling teoretis. Mengapa keduanya bukan penaksir keliling yang baik, dan konektivitas apa yang dimiliki masing-masing batas? (c) Gonzalez memakai SE silang pada Eq. 9-19. Jelaskan dengan angka TODO 3.3 apa yang terjadi bila kotak 3&times;3 dipakai, dan kaitkan dengan konektivitas kontur. (d) Mengapa jumlah komponen `objek_d` berbeda untuk konektivitas 8 dan 4, padahal `objek` sama?

*Jawaban:* ...

## Bagian 4 &mdash; Kerangka dan rekonstruksi morfologi (25 menit)

Kerangka (Eq. 9-28 s.d. 9-31): $S(A) = \bigcup_{k=0}^{K} S_k(A)$, $S_k(A) = (A \ominus kB) - (A \ominus kB) \circ B$, dengan $K$ langkah terakhir sebelum erosi menjadi kosong. Rekonstruksi dari kerangka (Eq. 9-32): $A = \bigcup_k (S_k(A) \oplus kB)$.

Dilasi geodesik (Eq. 9-38, 9-39): $D_G^{(1)}(F) = (F \oplus B) \cap G$, diulang sampai stabil menjadi rekonstruksi $R_G^D(F)$ (Eq. 9-42). Opening by reconstruction (Eq. 9-44): $O_R^{(n)}(F) = R_F^D(F \ominus nB)$.

In [ ]:
# TODO 4.1 — Kerangka Gonzalez dengan B = KOTAK3 pada `siluet`. Laporkan K, banyak piksel kerangka, jumlah komponen-8,
#            dan apakah rekonstruksi Eq. 9-32 sama persis dengan `siluet`. Bandingkan dengan skm.skeletonize dan skm.thin.
def kerangka(A, B=KOTAK3):
    """Mengembalikan (S, daftar S_k)."""
    ...

def rekonstruksi_kerangka(Sk, B=KOTAK3):
    ...


In [ ]:
# [Pengayaan] — Penipisan Gonzalez (Eq. 9-23, 9-25) dengan delapan SE Gbr. 9.23(a) (diberikan). Bandingkan banyak pikselnya
#              dengan skeletonize, lalu terapkan skm.thin pada hasilnya (pengganti konversi ke konektivitas-m).
SE_TIPIS = [_putar_cincin([[0, 0, 0], [-1, 1, -1], [1, 1, 1]], k) for k in range(8)]
def penipisan(A, seri=SE_TIPIS):
    X, p = A.copy(), 0
    while True:
        p += 1; X0 = X.copy()
        for B in seri: X = X & ~hmt(X, B)
        if np.array_equal(X, X0): return X, p
Tn, p = penipisan(siluet)
print('penipisan Gonzalez:', Tn.sum(), 'px dalam', p, 'putaran,', nkomp(Tn), 'komponen | setelah skm.thin:', skm.thin(Tn).sum(), 'px')


In [ ]:
# TODO 4.2 — Rekonstruksi dengan dilasi dan opening by reconstruction. Tujuan: ambil judul besar dari `teks`, buang badan teks kecil.
#            Bandingkan opening biasa dan opening by reconstruction dengan B = elips(5), lalu B = silang 5x5
#            (cv2.getStructuringElement(cv2.MORPH_CROSS, (5, 5))). Ukur piksel salah terhadap `judul` dan piksel badan teks yang tersisa.
def rekonstruksi_dilasi(F, G, B=KOTAK3):
    ...

def opening_rekonstruksi(F, B, n=1):
    ...


In [ ]:
# TODO 4.3 — Isi lubang otomatis (Eq. 9-45, 9-46) dan pembersihan tepi (Eq. 9-47, 9-48).
#   (a) isi_lubang_otomatis(I, B): penanda F = 1 − I pada tepi citra; H = komplemen R_{I^c}^D(F) dengan SE B.
#       Jalankan pada `cincin` dan `teks` dengan B = KOTAK3 (seperti Gonzalez Gbr. 9.32) dan B = SILANG. Laporkan piksel yang terisi.
#       Bandingkan dengan ndimage.binary_fill_holes.
#   (b) bersihkan_tepi(I): penanda = I pada tepi; X = I − R_I^D(F). Uji pada `sar`: berapa piksel yang terhapus?
def isi_lubang_otomatis(I, B=SILANG):
    ...

def bersihkan_tepi(I, B=KOTAK3):
    ...


In [ ]:
# [Pengayaan] TODO 4.4 — Pemangkasan (Eq. 9-34 s.d. 9-37). Fungsi `pangkas` diberikan. Buat siluet bertepi gerigi, ambil kerangkanya,
#            lalu laporkan banyak titik ujung untuk n = 3, 5, 10, 20.
SE_UJUNG = ([_putar_cincin([[-1, 0, 0], [1, 1, 0], [-1, 0, 0]], 2 * k) for k in range(4)] +
            [_putar_cincin([[1, 0, 0], [0, 1, 0], [0, 0, 0]], 2 * k) for k in range(4)])
def pangkas(A, n=3):
    X1 = A.copy()
    for _ in range(n):
        for B in SE_UJUNG: X1 = X1 & ~hmt(X1, B)
    X2 = np.zeros(A.shape, bool)
    for B in SE_UJUNG: X2 |= hmt(X1, B)
    X3 = X2.copy()
    for _ in range(n): X3 = dilasi(X3, KOTAK3) & A
    return X1 | X3
rng5 = np.random.default_rng(5)
Kn = siluet ^ (batas(siluet) & (rng5.random(siluet.shape) < 0.3))
Kn = Kn | (dilasi(siluet, KOTAK3) & ~siluet & (rng5.random(siluet.shape) < 0.15))
Kn = opening_rekonstruksi(Kn, KOTAK3)
skn = skm.skeletonize(Kn)
print('kerangka bergerigi:', skn.sum(), 'px,', ujung(skn), 'titik ujung (siluet mulus:', ujung(skm.skeletonize(siluet)), ')')


**Analisis 4.** (a) Kerangka Eq. 9-28 dapat merekonstruksi siluet dengan tepat, tetapi tidak terhubung. Dengan angka TODO 4.1, jelaskan mengapa sifat pertama tidak menjamin sifat kedua, dan mengapa "panjang kerangka" sebagai fitur bentuk wajib menyebut metodenya. (b) Mengapa opening biasa dengan elips 5&times;5 merusak judul, sedangkan opening by reconstruction tidak? Dengan silang 5&times;5, rekonstruksi justru menyisakan badan teks. Apa syarat penanda yang baik? (c) Dengan angka TODO 4.3, jelaskan mengapa Eq. 9-46 dengan SE 3&times;3 gagal mengisi lubang belah ketupat, lingkaran, dan huruf, sedangkan silang berhasil. Rumuskan aturan konektivitas objek&ndash;latar.

*Jawaban:* ...

## Bagian 5 &mdash; Studi kasus: membersihkan masker deteksi kapal dari citra SAR (20 menit)

Citra SAR (*synthetic aperture radar*) sintetis memuat sembilan kapal terang di laut berspekel, daratan di pojok kiri atas, dua kapal bersandingan, dan satu kapal terpotong tepi kanan. Pengambangan global (2,6 &times; rerata laut, gaya CFAR sederhana) menghasilkan `sar`: kapal terdeteksi, tetapi ribuan piksel spekel ikut lolos dan kapal berlubang.

Tujuan: masker berisi **delapan kapal utuh** (indeks 0&ndash;7) tanpa daratan, spekel, dan kapal terpotong. Fungsi penilaian diberikan. **Peringatan:** citra sintetis; angka tidak boleh dipakai untuk menyimpulkan kinerja pada citra Sentinel-1 atau sensor lain.

In [ ]:
# Diberikan — penilaian
utuh = per_kapal[:8]; gt = np.any(utuh, axis=0)
def nilai_sar(R):
    lab, n = ndimage.label(R, np.ones((3, 3)))
    det = [bool(R[m].mean() >= 0.5) for m in utuh]
    luas = [int((R & dilasi(m, np.ones((5, 5), int))).sum()) for m in utuh]
    galat = [100 * abs(a - m.sum()) / m.sum() for a, m, d in zip(luas, utuh, det) if d]
    palsu = sum(1 for j in range(1, n + 1) if not (gt & (lab == j)).any())
    return dict(komp=n, terdeteksi=f'{sum(det)}/8', komp_palsu=palsu, iou=round(float((R & gt).sum() / (R | gt).sum()), 3),
                galat_luas=round(float(np.mean(galat)), 1), luas_kapal5=luas[5])
print('masukan:', nilai_sar(sar), '| luas kapal:', [int(m.sum()) for m in per_kapal])


In [ ]:
# TODO 5.1 — Bangun dan nilai pipa berikut dengan fungsi Anda sendiri:
#   P1 opening kotak3 | P2 opening kotak5 | P3 opening by reconstruction kotak3 | P4 = P3 + isi lubang (silang)
#   P5 = P4 + bersihkan tepi | P6 = bersihkan_tepi(isi_lubang(closing(P1, kotak3))) | P7 = P6 tanpa komponen < 15 piksel
#   Cetak hasil nilai_sar untuk P0 (= sar) sampai P7 dan tampilkan P2, P5, P7.
P = {'P0': sar}


In [ ]:
# TODO 5.2 — Diagnosis. (a) Kapal mana yang hilang di P5? Tunjukkan dengan label komponen P4 bahwa kapal itu satu komponen
#            dengan daratan. (b) Erosi berulang (kotak3) masker gabungan dua kapal bersandingan (per_kapal[6] | per_kapal[7]):
#            setelah berapa erosi ia terpisah menjadi dua, atau lenyap tanpa pernah terpisah?


**Analisis 5.** (a) Pipa mana yang Anda rekomendasikan? Justifikasi dengan terdeteksi, komponen palsu, IoU, dan galat luas sekaligus. (b) Opening by reconstruction (P3) menjaga luas kapal paling tepat. Mengapa P5 yang memakainya justru kehilangan satu kapal? (c) Ambang luas 15 piksel pada P7 terletak di antara 12 piksel (serpihan daratan) dan 22 piksel (kapal terkecil setelah opening). Apa risiko memilih ambang itu dari citra ini sendiri? (d) Dua kapal bersandingan selalu terhitung satu. Mengapa erosi berulang tidak dapat memisahkannya, dan teknik apa (Sub-CPMK-7) yang diperlukan? (e) Apakah membuang kapal terpotong di tepi selalu benar untuk tim SAR?

*Jawaban:* ...

## Bagian 6 &mdash; Pemeriksaan akhir dan pengumpulan (5 menit)

- [ ] Semua sel berjalan tanpa galat dari atas (Restart & Run All).
- [ ] Uji `erosi`/`dilasi` (3 dan 35 piksel; sama dengan SciPy), `hmt` (lubang dan sudut), dan `rekonstruksi_dilasi` lolos.
- [ ] Tabel Bagian 2 (5 SE &times; 2 urutan) dan tabel pipa Bagian 5 (P0&ndash;P7) lengkap.
- [ ] Setiap blok Analisis memuat angka dari keluaran sendiri; penggunaan alat bantu AI dicantumkan.
- [ ] Berkas dinamai `PraktikumPCD_SubCPMK6_NIM_Nama.ipynb` dan diunggah ke LMS. Hasil praktikum ini masuk Laporan Praktikum Blok 2.